In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

titanic = sns.load_dataset("titanic")
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
df = titanic.drop(columns=[
    'pclass', 'sibsp', 'parch', 'fare', 'class',
    'who', 'adult_male', 'deck', 'embark_town',
    'alive', 'alone'
])

df = df.dropna()

df = pd.get_dummies(df, columns=['sex', 'embarked'])

df.head()

,survived,age,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,0,22.0,False,True,False,False,True
1,1,38.0,True,False,True,False,False
2,1,26.0,True,False,False,False,True
3,1,35.0,True,False,False,False,True
4,0,35.0,False,True,False,False,True


In [3]:
X = df.drop(columns=['survived'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, shuffle=True, random_state=42
)

In [4]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[103  19]
 [ 34  58]]
              precision    recall  f1-score   support

           0       0.75      0.84      0.80       122
           1       0.75      0.63      0.69        92

    accuracy                           0.75       214
   macro avg       0.75      0.74      0.74       214
weighted avg       0.75      0.75      0.75       214



In [5]:
passenger = [
    22, # age
    0,  # sex_female
    1,  # sex_male
    0,  # embarked_C
    0,  # embarked_Q
    1   # embarked_S
]

predict_passenger = model.predict(
    pd.DataFrame([passenger], columns=X.columns)
)

predict_passenger

array([0])

In [6]:
# Compare against original passengers
matched = titanic[
    (titanic['age'] == 22) &
    (titanic['sex'] == 'male') &
    (titanic['embarked'] == 'S')
]

print(matched['survived'].value_counts())

# Compare embarkation cities
cities = pd.DataFrame([
    [22, 0, 1, 1, 0, 0],
    [22, 0, 1, 0, 1, 0],
    [22, 0, 1, 0, 0, 1]
], columns=X.columns, index=['Cherbourg', 'Queenstown', 'Southampton'])

print(pd.DataFrame({
    'Prediction': model.predict(cities),
    'Survival %': (model.predict_proba(cities)[:, 1] * 100).round(1)
}, index=cities.index))

survived
0    12
Name: count, dtype: int64
             Prediction  Survival %
Cherbourg             0        32.4
Queenstown            0        13.8
Southampton           0        16.9


In [7]:
print(titanic["survived"].value_counts())

survived
0    549
1    342
Name: count, dtype: int64
